In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [2]:
%store -r data
data=data
%store -r preprocessor
preprocessor=preprocessor

In [3]:
# Prepare the full feature set
# x_content is the feature set including content-based features
X_content = preprocessor.fit_transform(data) # feature vector for al auto
X_content.shape

(29, 105)

In [4]:

cosine_sim_content = cosine_similarity(X_content, X_content)
cosine_sim_item = cosine_similarity(X_content)

#cosine_sim
#print(cosine_sim)

In [5]:
def recommend_contentbased_with_score(car_index, top_n=5):
    similarity_scores = list(enumerate(cosine_sim_content[car_index]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    top_items = similarity_scores[1:top_n+1]

    df_rec = data.iloc[[i for i, _ in top_items]].copy()
    df_rec['item_id'] = df_rec.index
    df_rec['cb_score'] = [score for _, score in top_items]

    return df_rec[['item_id', 'cb_score']]


In [6]:
def recommend_collaborative_with_score(car_index, top_n=5):
    similarity_scores = list(enumerate(cosine_sim_cf[car_index]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    top_items = similarity_scores[1:top_n+1]

    df_rec = data.iloc[[i for i, _ in top_items]].copy()
    df_rec['item_id'] = df_rec.index
    df_rec['cf_score'] = [score for _, score in top_items]

    return df_rec[['item_id', 'cf_score']]


In [7]:
def recommend_hybrid(car_index, top_n=5, alpha=0.5):
    """
    alpha = Gewicht für Collaborative Filtering (0–1)
    (1 - alpha) = Gewicht für Content-Based
    """

    # Content-Based
    cb = recommend_contentbased_with_score(car_index, top_n)

    # Collaborative Filtering
    cf = recommend_collaborative_with_score(car_index, top_n)

    # Merge auf item_id
    hybrid = pd.merge(cb, cf, on='item_id', how='outer')

    # NaNs ersetzen
    hybrid[['cb_score', 'cf_score']] = hybrid[['cb_score', 'cf_score']].fillna(0)

    # Hybrid Score berechnen
    hybrid['hybrid_score'] = (
        alpha * hybrid['cf_score']
        + (1 - alpha) * hybrid['cb_score']
    )

    # Sortieren & Top-N
    hybrid = hybrid.sort_values(
        by='hybrid_score',
        ascending=False
    ).head(top_n)

    # Finale Autos zurückgeben
    return data.loc[hybrid['item_id']]


In [8]:
recommended_cars = recommend_hybrid(car_index=0, top_n=5, alpha=0.6)
recommended_cars[['Brand', 'Model', 'year of manufacture']]


NameError: name 'cosine_sim_cf' is not defined